In [7]:
import pandas as pd
import json


ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

ledger = ledger.drop_duplicates()
gateway = gateway.drop_duplicates()

comparison = pd.merge(ledger, gateway, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'))

missing_in_gateway = comparison[comparison['amount_usd_gateway'].isna()]
missing_in_ledger = comparison[comparison['amount_usd_ledger'].isna()]
amount_mismatches = comparison[(comparison['amount_usd_ledger'] != comparison['amount_usd_gateway']) &
                               comparison['amount_usd_ledger'].notna() &
                               comparison['amount_usd_gateway'].notna()]

missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('amount_mismatches.csv', index=False)

print("✅ Mismatch files created!")

✅ Mismatch files created!


In [8]:
with open('api_response_sample.json') as f:
    json_data = json.load(f)

df_api = pd.json_normalize(json_data)

df_api.to_csv('api_normalized.csv', index=False)
print("✅ JSON is now a clean CSV!")

✅ JSON is now a clean CSV!


In [9]:
status_mismatches = comparison[
    (comparison['status_ledger'] != comparison['status_gateway']) &
    comparison['status_ledger'].notna() &
    comparison['status_gateway'].notna()
]
status_mismatches.to_csv('status_mismatches.csv', index=False)

reconciliation_report = pd.concat([
    missing_in_gateway.assign(issue_type='Missing in Gateway'),
    missing_in_ledger.assign(issue_type='Missing in Ledger'),
    amount_mismatches.assign(issue_type='Amount Mismatch'),
    status_mismatches.assign(issue_type='Status Mismatch')
])

reconciliation_report.to_csv('reconciliation_report.csv', index=False)

print("✅ status_mismatches.csv and reconciliation_report.csv are ready!")

✅ status_mismatches.csv and reconciliation_report.csv are ready!


In [25]:
ledger.groupby('transaction_date')['amount_usd'].sum().to_csv('daily_summary.csv')

In [28]:
ledger.groupby('payment_method')['amount_usd'].sum().to_csv('payment_method_breakdown.csv')

In [31]:
ledger.groupby('merchant_id').agg({'amount_usd':'sum', 'transaction_id':'count'}).to_csv('merchant_performance_summary.csv')

In [34]:
#not able to generate region_method_breakdon.csv